# 01 - Venue Data & Schema Inspection

This notebook inspects the raw-to-interim extracted High-Volume FHV records for January 2024.
It validates:
- Schema and column data types
- Null values
- Pickup location distributions across Barclays Center, Citi Field, and designated spillover zones.

In [2]:
from pathlib import Path
import pandas as pd

# Define path to the interim filtered parquet file
DATA_PATH = Path('../data/interim/venue_trips_2024-01.parquet')
if not DATA_PATH.exists():
    # Fallback if executing from root directory
    DATA_PATH = Path('data/interim/venue_trips_2024-01.parquet')

print(f'Loading data from: {DATA_PATH.resolve()}')
df = pd.read_parquet(DATA_PATH)
print(f'Successfully loaded {len(df):,} records.')

Loading data from: C:\Users\rafid\Desktop\Projects\surge-regressor\data\interim\venue_trips_2024-01.parquet
Successfully loaded 19,663,930 records.


### 1. Shape and Schema Overview

In [3]:
print('Dataset Dimensions (Rows, Columns):', df.shape)
print('\n--- Column Data Types ---')
df.info()

Dataset Dimensions (Rows, Columns): (19663930, 24)

--- Column Data Types ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19663930 entries, 0 to 19663929
Data columns (total 24 columns):
 #   Column                Dtype         
---  ------                -----         
 0   hvfhs_license_num     object        
 1   dispatching_base_num  object        
 2   originating_base_num  object        
 3   request_datetime      datetime64[us]
 4   on_scene_datetime     datetime64[us]
 5   pickup_datetime       datetime64[us]
 6   dropoff_datetime      datetime64[us]
 7   PULocationID          int32         
 8   DOLocationID          int32         
 9   trip_miles            float64       
 10  trip_time             int64         
 11  base_passenger_fare   float64       
 12  tolls                 float64       
 13  bcf                   float64       
 14  sales_tax             float64       
 15  congestion_surcharge  float64       
 16  airport_fee           float64       
 17  tips

### 2. Missing Value Analysis

In [4]:
null_summary = pd.DataFrame({
    'Missing Values': df.isna().sum(),
    'Percentage (%)': (df.isna().sum() / len(df) * 100).round(2)
})
null_summary

,Missing Values,Percentage (%)
hvfhs_license_num,0,0.00
dispatching_base_num,0,0.00
originating_base_num,5218737,26.54
request_datetime,0,0.00
on_scene_datetime,5218737,26.54
pickup_datetime,0,0.00
dropoff_datetime,0,0.00
PULocationID,0,0.00
DOLocationID,0,0.00
trip_miles,0,0.00


### 3. Target Zone Breakdown
- **Barclays Center:** Core = `189`, Spillovers = `[49, 61, 62, 97, 181, 190]`
- **Citi Field:** Core = `253`, Spillover = `[92, 93]`

In [ ]:
zone_labels = {
    189: 'Barclays Core (Prospect Heights)',
    49: 'Barclays Spillover (Clinton Hill)',
    61: 'Barclays Spillover (Crown Heights North)',
    62: 'Barclays Spillover (Crown Heights South)',
    97: 'Barclays Spillover (Fort Greene)',
    181: 'Barclays Spillover (Park Slope)',
    190: 'Barclays Spillover (Prospect Park)',
    253: 'Citi Field Core (Willets Point)',
    92: 'Citi Field Spillover (Flushing)',
    93: 'Citi Field Spillover (Flushing Meadows-Corona Park)'
}

zone_counts = df.loc[df['PULocationID'].isin(zone_labels), 'PULocationID'].value_counts().sort_index().reset_index()
zone_counts.columns = ['PULocationID', 'Trip_Count']
zone_counts['Zone_Description'] = zone_counts['PULocationID'].map(zone_labels)
zone_counts['Share (%)'] = (zone_counts['Trip_Count'] / len(df) * 100).round(2)
zone_counts[['PULocationID', 'Zone_Description', 'Trip_Count', 'Share (%)']]

,PULocationID,Zone_Description,Trip_Count,Share (%)
0,49,Barclays Spillover (Clinton Hill),117734,0.60
1,61,Barclays Spillover (Crown Heights North),256556,1.30
2,62,Barclays Spillover (Crown Heights South),72028,0.37
3,92,Citi Field Spillover (Flushing),97103,0.49
4,93,Citi Field Spillover (Flushing Meadows-Corona ...,5350,0.03
5,97,Barclays Spillover (Fort Greene),121814,0.62
6,181,Barclays Spillover (Park Slope),191073,0.97
7,189,Barclays Core (Prospect Heights),73051,0.37
8,190,Barclays Spillover (Prospect Park),8460,0.04
9,253,Citi Field Core (Willets Point),430,0.00


### 4. Sample Record Preview

In [6]:
df.head(10)

,hvfhs_license_num,dispatching_base_num,originating_base_num,request_datetime,on_scene_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,...,sales_tax,congestion_surcharge,airport_fee,tips,driver_pay,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag
0,HV0003,B03404,B03404,2024-01-01 00:21:47,2024-01-01 00:25:06,2024-01-01 00:28:08,2024-01-01 01:05:39,161,158,2.83,...,4.05,2.75,0.0,0.0,40.18,N,N,N,N,N
1,HV0003,B03404,B03404,2024-01-01 00:10:56,2024-01-01 00:11:08,2024-01-01 00:12:53,2024-01-01 00:20:05,137,79,1.57,...,0.89,2.75,0.0,0.0,6.12,N,N,N,N,N
2,HV0003,B03404,B03404,2024-01-01 00:20:04,2024-01-01 00:21:51,2024-01-01 00:23:05,2024-01-01 00:35:16,79,186,1.98,...,1.60,2.75,0.0,0.0,9.47,N,N,N,N,N
3,HV0003,B03404,B03404,2024-01-01 00:35:46,2024-01-01 00:39:59,2024-01-01 00:41:04,2024-01-01 00:56:34,234,148,1.99,...,1.52,2.75,0.0,0.0,11.35,N,N,N,N,N
4,HV0003,B03404,B03404,2024-01-01 00:48:19,2024-01-01 00:56:23,2024-01-01 00:57:21,2024-01-01 01:10:02,148,97,2.65,...,3.43,2.75,0.0,0.0,28.63,N,N,N,N,N
5,HV0003,B03404,B03404,2024-01-01 00:03:47,2024-01-01 00:05:53,2024-01-01 00:06:15,2024-01-01 00:27:53,255,95,7.02,...,2.85,0.00,0.0,0.0,24.35,N,N,N,N,Y
6,HV0003,B03404,B03404,2024-01-01 00:22:51,2024-01-01 00:29:17,2024-01-01 00:29:47,2024-01-01 00:50:08,95,212,11.33,...,4.68,0.00,0.0,0.0,30.98,N,N,N,N,Y
7,HV0003,B03404,B03404,2024-01-01 00:45:34,2024-01-01 00:57:29,2024-01-01 00:57:50,2024-01-01 01:11:27,213,47,3.43,...,2.06,0.00,0.0,0.0,20.73,N,N,N,N,Y
8,HV0003,B03404,B03404,2024-01-01 00:11:51,2024-01-01 00:15:46,2024-01-01 00:16:00,2024-01-01 00:28:13,209,114,1.54,...,1.37,2.75,0.0,0.0,10.40,N,N,N,N,Y
9,HV0003,B03404,B03404,2024-01-01 00:26:48,2024-01-01 00:33:02,2024-01-01 00:33:15,2024-01-01 00:46:39,113,209,1.72,...,1.21,2.75,0.0,0.0,11.38,N,N,N,N,Y
